In [2]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 22.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 62.8 MB/s eta 0:00:00


In [41]:
from transformers import pipeline
import os
import sys
import argparse
from glob import glob
from pathlib import Path
from shutil import copy2

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [43]:
#this part is the parameters part
output = 'drive/MyDrive/Colab Notebooks/abstract_2_output'
workdir=""
input = 'drive/MyDrive/Colab Notebooks/abstracts_2'
checkpoint = "drive/MyDrive/Colab Notebooks/debbie-classifier-model-transformers-1.0.0"

In [29]:
#create output folder if not exist
if not os.path.exists(output):
    os.makedirs(output)


In [30]:
pipe = pipeline("text-classification", model=checkpoint) 
print("create pipeline done")

create pipeline done


In [44]:
#get subfolders that my contain txt files
folders = glob(input + '/**/', recursive=True)
#add root in folders
#folders.insert(0, args.i)

#set counters
total=0
total_relevant=0
total_non_relevant=0

#iterate over folders and classify
for f in folders:
    print ('folder:' + f)
    test_set = glob(f + '/*.txt', recursive=True)
    if(len(test_set)>0):
        #extract features new abstracts
        #test_count = debbie_vectorizer.transform(test_set)
        #test_tfidf = debbie_transformer.transform(test_count)
        #Read text from file
        X_test = [Path(file).read_text() for file in test_set]
        #Apply corresponding preprocessing
        
        #run model
        y_predict = pipe(X_test, padding=True, truncation=True)
        #for index,abstracts in enumerate(X_test):
            
        # the results of the classification
        results = dict(zip(test_set, y_predict))
        print(results)
        relevant_abstracts = []
        not_relevant_abstracts = []

        for key, value in results.items():
            if value['label'] == 'clinical':
                print(value)
                relevant_abstracts.append(key)
                filepathTokens = key.split('/')
                lastToken = filepathTokens[len(filepathTokens) - 1]
                copy2(key, output)
                meta=key.replace('.txt','.meta')
                metafile=lastToken.replace('.txt','.meta')
                copy2(meta, output)
                with open (meta, 'r+') as f, open(os.path.join(output, metafile), 'r+') as c:
                    content = f.read()
                    c.write( content  +  'clinical' + '\n')

            elif value['label'] == 'non-clinical':
                relevant_abstracts.append(key)
                filepathTokens = key.split('/')
                lastToken = filepathTokens[len(filepathTokens) - 1]
                copy2(key, output)
                meta=key.replace('.txt','.meta')
                metafile=lastToken.replace('.txt','.meta')
                copy2(meta, output)
                with open (meta, 'r+') as f, open(os.path.join(output, metafile), 'r+') as c:
                    content = f.read()
                    c.write( content +  'non-clinical' + '\n')
            else:
                print(value) 
                not_relevant_abstracts.append(key)

        #save relevant abstracts to folder
        # for filename in test_set:
        #     if str(filename) in relevant_abstracts:
        #         copy2(filename, args.o)

        print('total number of abstracts collected in folder', (len(relevant_abstracts)+len(not_relevant_abstracts)))
        print('number of relevant abstracts in folder:', len(relevant_abstracts))
        print('number of non-relevant abstracts in folder:', len(not_relevant_abstracts))
        total =  total  + len(relevant_abstracts)+len(not_relevant_abstracts)
        total_relevant =  total_relevant  + len(relevant_abstracts)
        total_non_relevant =  total_non_relevant  + len(not_relevant_abstracts)
    else:
        print('Emtpy Folder')
print('+++++++++++++++++++++++++++++++++++++++++++++++++++++')
print('total number of abstracts collected:', total)
print('number of relevant abstracts:', total_relevant)
print('number of non-relevant abstracts:', total_non_relevant)

folder:drive/MyDrive/Colab Notebooks/abstracts_2/
{'drive/MyDrive/Colab Notebooks/abstracts_2/35776300.txt': {'label': 'non-relevant', 'score': 0.5815798044204712}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35917216.txt': {'label': 'non-clinical', 'score': 0.9999464750289917}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35829772.txt': {'label': 'non-clinical', 'score': 0.9977055191993713}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35819543.txt': {'label': 'non-clinical', 'score': 0.9988842606544495}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35709532.txt': {'label': 'non-relevant', 'score': 0.9999707937240601}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35758934.txt': {'label': 'non-relevant', 'score': 0.7363117337226868}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35803460.txt': {'label': 'clinical', 'score': 0.9956846237182617}, 'drive/MyDrive/Colab Notebooks/abstracts_2/35863877.txt': {'label': 'non-relevant', 'score': 0.9999804496765137}, 'drive/MyDrive/Colab Notebooks/abstracts_